# A2. 국내 검색 관심 — 할인과 출시는 한국 유저의 관심을 얼마나 끌어올리나

* 담당: 김준영 · 분석 방향 문서의 **A2** (11/22 회의 확정)
* 한 줄 질문: **"얼마나 깎아야 한국 유저의 관심이 움직이고, 출시 화제성은 얼마나 빨리 식나?"**
* 사업 연결: 퍼블리셔에게 *관심이 눈에 띄게 늘기 시작하는 할인율 구간*을 제시하고, *관심이 몰리는 시기*를 광고 노출 시점의 근거로 쓴다.

### 데이터 처리 과정 (이 프로젝트의 수집 → 전달 → 정제 → 저장 → 활용 중 이 노트북이 맡는 부분)

| 단계 | 내용 | 이 노트북에서 |
| --- | --- | --- |
| 수집 | ITAD 가격 이력, IGDB 출시일 (이미 파이프라인이 수집·정제·MariaDB 저장) | SQL로 읽기 전용 추출한 TSV를 불러옴 |
| 수집 | **네이버 데이터랩 검색어 트렌드** (파이프라인에 없는 분석 전용 외부 데이터) | API를 직접 호출하고 결과를 `analysis/data/`에 캐시 |
| 정제 | 스토어별 할인율 → 일별 최고 할인율, 할인 시작일 추출, 검색량 척도 보정 | 4장 |
| 활용 | 가설 검정 → 인사이트 | 5~8장 |

### 분석 순서
1. 환경 설정 → 2. **가설 설정** → 3. 데이터 불러오기 → 4. 데이터 점검·전처리 → 5. 탐색(EDA) → 6. **통계 검정** → 7. 의미 있는 결과인가? → 8. **인사이트·제안**

> 이 노트북의 숫자는 모두 실행 결과로 계산됩니다. 해설 문장 중 "확인할 수 있다"는 *무엇을 보는 차트인지* 설명이며, 결론은 7~8장에서 실제 값으로 자동 출력됩니다.

### 준비 (한 번만)
```bash
pip install pandas numpy scipy seaborn matplotlib statsmodels requests jupyterlab
```
* DB 데이터 추출 (mariadb-re, 읽기 전용 계정):
```bash
mysql -u <읽기 계정> -p gamenuri -B --default-character-set=utf8mb4 < analysis/scripts/a2_games.sql         > analysis/data/a2_games.tsv
mysql -u <읽기 계정> -p gamenuri -B --default-character-set=utf8mb4 < analysis/scripts/a2_price_history.sql > analysis/data/a2_price_history.tsv
```
* 네이버 키는 **코드에 쓰지 않습니다.** 아래 순서로 찾습니다. (캐시가 있으면 키가 필요 없습니다.)
  1. 환경 변수 `NAVER_CLIENT_ID`, `NAVER_CLIENT_SECRET`
  2. 저장소 루트의 `.env` 파일(Git에 올라가지 않음)에 `NAVER_CLIENT_ID=...`, `NAVER_CLIENT_SECRET=...` 두 줄
  3. 없으면 3-2의 **두 번째 코드 셀**(요청마다 API를 호출하는 셀)을 실행할 때 입력창이 **한 번** 뜹니다 (VS Code에서는 화면 맨 위 가운데, Jupyter에서는 셀 아래)
  * 이 노트북은 **NAVER API HUB**(네이버 클라우드) 키를 기준으로 합니다. 이 키는 Application을 등록할 때 **'검색어 트렌드' API를 함께 선택**해야 동작합니다. 개발자센터(developers.naver.com)에서 예전에 받은 키라면 3-2 코드의 `NAVER_PLATFORM`을 `'legacy'`로 바꿉니다. 두 키는 서로 호환되지 않습니다.

## 1. 환경 설정

In [ ]:
# 불필요한 경고 메시지를 숨겨 출력이 깔끔하게 보이도록 합니다.
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# 분석에 필요한 라이브러리를 불러옵니다.
# pandas/numpy: 데이터 처리, scipy/statsmodels: 통계 검정, matplotlib/seaborn: 시각화, requests: 네이버 API 호출
import os, json, time, hashlib, getpass
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
from scipy import stats
from statsmodels.stats.multitest import multipletests

%matplotlib inline

### 한글 폰트 설정

In [ ]:
# 그래프의 한글이 깨지지 않도록 설치된 한글 폰트를 찾아 지정합니다. (Windows: 맑은 고딕, Mac: AppleGothic)
candidates = ['Malgun Gothic', 'AppleGothic', 'NanumGothic', 'Noto Sans CJK KR', 'Noto Sans CJK JP']
available = {f.name for f in fm.fontManager.ttflist}
font_name = next((c for c in candidates if c in available), None)

plt.rcParams['font.family'] = font_name or 'sans-serif'
plt.rcParams['axes.unicode_minus'] = False     # 음수 부호 깨짐 방지
plt.rcParams['figure.dpi'] = 110
sns.set_style('whitegrid', {'font.family': plt.rcParams['font.family']})
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', lambda v: f'{v:,.3f}')

BLUE, ORANGE, GRAY, GREEN = '#2a78d6', '#eb6834', '#a3a29d', '#2f9e6e'
print('사용 폰트:', font_name)

from matplotlib.ticker import FuncFormatter
def plain_log(ax, axis='y'):
    """로그축 눈금을 10⁻¹ 대신 0.1, 1, 10 으로 표시합니다 (한글 폰트에서 위첨자 마이너스가 깨지는 것을 피함)."""
    def minor(v, _):
        lead = round(v / 10 ** np.floor(np.log10(v))) if v > 0 else 0
        return f'{v:g}' if lead in (2, 5) else ''
    t = ax.yaxis if axis == 'y' else ax.xaxis
    t.set_major_formatter(FuncFormatter(lambda v, _: f'{v:g}'))
    t.set_minor_formatter(FuncFormatter(minor))

## 2. 가설 설정

데이터를 보기 **전에** 가설과 판정 기준을 먼저 정해 둡니다. (결과를 본 뒤 기준을 바꾸면 검정이 의미가 없어집니다.)

| # | 가설 (H1: 우리가 기대하는 것) | 귀무가설 (H0) | 통계 도구 | 실무 기준 ("의미 있다"의 조건) |
| --- | --- | --- | --- | --- |
| H1 | 할인이 시작되면 **한국 검색 관심이 늘어난다** (시작 전 7일 < 시작 후 7일) | 할인 전후 검색량 변화가 0이다 | Shapiro-Wilk로 정규성 확인 → **Wilcoxon 부호순위**(단측), 위약(placebo) 날짜와 **Mann-Whitney U** 비교 | 중앙값 +20% 이상 |
| H2 | **할인율이 클수록 관심 증가폭도 크다.** 일정 할인율 아래에서는 효과가 거의 없다 | 할인율 구간 사이에 차이가 없다 | **Spearman 상관**, **Kruskal-Wallis**, 구간별 Wilcoxon (**Holm 보정**) | 구간 중앙값 +20% 이상이면서 유의 |
| H3 | **출시 화제성은 3주(21일) 안에 최고점의 절반으로 식는다** | 절반이 되는 데 21일보다 오래 걸린다 | **Kaplan-Meier**(중도절단 처리), **이항검정** | 반감 중앙값 ≤ 21일 |

* **유의수준 α = 0.05.** 통계적으로 유의해도 효과가 작으면 "의미 없음"으로 판정합니다 (p값만 보지 않음).
* **위약(placebo) 대조:** 할인과 상관없는 보통 날의 7일 변화도 계산해서, "원래 이 정도는 출렁인다"와 비교합니다. 네이버 검색량은 요일·시즌 요인이 있어 전후 비교만으로는 부족하기 때문입니다.
* **사후 보강(투명성):** 1차 실행(게임 60개) 결과를 본 뒤 (1) 요일 효과 통제, (2) 판정을 3단계(일치 / 부분 일치 / 불일치)로 표기, (3) 표본 확대(`TOP_N` 60 → 150)를 추가했습니다. **가설과 실무 기준(+20%)은 바꾸지 않았습니다.** 표본을 늘릴 때 게임은 결과가 아니라 한국어 리뷰 수 순으로 뽑았습니다.
* **같은 게임의 이벤트는 독립이 아니므로** 신뢰구간은 *게임 단위로 다시 뽑는(cluster bootstrap)* 방식으로 계산합니다.

### 분석 설정값 (한 곳에서 관리)

In [ ]:
# ---- 데이터 위치 ----
DATA_DIR = Path(os.environ.get('GAMENURI_DATA_DIR', '../data'))     # analysis/data (Git에 올리지 않음)
RESULT_DIR = Path(os.environ.get('GAMENURI_RESULT_DIR', '../results/a2'))
CACHE_DIR = DATA_DIR / 'naver_cache'
for d in (RESULT_DIR, CACHE_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ---- 분석 대상 ----
TOP_N = 150                 # 한국어 리뷰가 많은 순으로 몇 개 게임을 볼지 (1차 실행은 60개, 표본 확대를 위해 150개로 늘림. 네이버 호출 횟수 = TOP_N/4 + 1 ≈ 39회, 일일 한도 1,000회)
EXCLUDE_IDS = []            # 검색어가 일반 단어라 제외할 game_id (3-1 점검 후 채움)
KEYWORD_OVERRIDE = {}       # {game_id: ['검색어1', '검색어2']} 한글명이 모호할 때 직접 지정
ANCHOR_NAME, ANCHOR_KEYWORDS = '스팀', ['스팀', 'steam']   # 요청끼리 척도를 맞추는 기준 검색어

# ---- 할인 이벤트 정의 ----
WINDOW = 7                  # 전후 비교 기간(일)
MIN_DISC = 10               # 이 % 이상이면 "할인 중" (SALE_MIN_PCT와 같은 값)
LAUNCH_EXCLUDE_DAYS = 30    # 출시 후 30일 안의 할인은 출시 화제성과 섞여서 제외
PLACEBO_PER_GAME = 10       # 게임마다 뽑는 "할인 없는 날" 수
MIN_LEVEL = 0.5             # 평균 검색량(스팀=100 기준)이 이보다 작으면 잡음이라 제외
PEAK_MARGIN = 5             # 요일 보정 후 곡선에서 '정점'이라고 부르려면 첫 주 평균보다 이 지수 포인트 이상 높아야 함

# ---- 통계 ----
ALPHA = 0.05
PRACTICAL = 1.2             # 실무 기준: 검색량이 1.2배(+20%) 이상
PRACTICAL_LOG2 = np.log2(PRACTICAL)
DISC_BINS = [10, 30, 50, 70, 101]
DISC_LABELS = ['10~29%', '30~49%', '50~69%', '70%~']
DECAY_DAYS = 21             # H3 기준: 3주
N_BOOT, SEED = 2000, 42

R = {}                      # 최종 요약에 쓸 결과 저장소

def savefig(fig, name):
    fig.tight_layout()
    fig.savefig(RESULT_DIR / name, dpi=150, bbox_inches='tight')

## 3. 데이터 불러오기

### 3-1) DB에서 뽑은 게임·가격 이력

In [ ]:
# mysql --batch로 뽑은 TSV를 읽습니다. 값이 없는 칸은 문자열 'NULL'로 나오므로 결측치로 바꿔 줍니다.
def read_tsv(name, **kw):
    path = DATA_DIR / name
    if not path.exists():
        raise FileNotFoundError(f'{path} 가 없습니다. 위 "준비" 칸의 mysql 명령으로 먼저 추출하세요.')
    return pd.read_csv(path, sep='\t', na_values=['NULL'], **kw)

games = read_tsv('a2_games.tsv', parse_dates=['release_date'])
prices = read_tsv('a2_price_history.tsv', parse_dates=['collected_at'])
print('games  :', games.shape)
print('prices :', prices.shape)
games.head()

In [ ]:
# 가격 이력 샘플 — 가격이 바뀐 시점마다 한 줄씩 쌓여 있습니다.
prices.head()

In [ ]:
# 분석 대상 게임 선택: 한글 이름 3글자 이상 + 한국어 리뷰가 많은 순 TOP_N개
games['name_ko_len'] = games['name_ko'].fillna('').str.len()
cand = games[(games['name_ko_len'] >= 3) & (~games['game_id'].isin(EXCLUDE_IDS))]
cand = cand.sort_values('review_count_ko', ascending=False).head(TOP_N)
cand = cand[cand['game_id'].isin(prices['game_id'].unique())]     # 가격 이력이 있는 게임만
game_ids = cand['game_id'].tolist()
release = cand.set_index('game_id')['release_date'].to_dict()
names = cand.set_index('game_id')['name_ko'].to_dict()
print(f'분석 대상 게임: {len(game_ids)}개')
cand[['game_id', 'name_ko', 'release_date', 'review_count_ko']].head(10)

### 3-2) 네이버 데이터랩 — 날짜별 검색 비율

네이버는 검색 *횟수*가 아니라 **요청 안에서 가장 높은 값을 100으로 한 상대값**을 줍니다. 그래서 그대로 모으면 요청끼리, 게임끼리 비교가 안 됩니다. 해결:

1. 모든 요청에 **기준 검색어("스팀")** 를 함께 넣는다. (요청당 키워드 그룹은 최대 5개 → 기준 1 + 게임 4)
2. 요청마다 기준 검색어의 평균이 100이 되도록 그 요청 전체를 같은 비율로 곱해 준다. → 모든 게임이 **"스팀 검색량 평균 = 100"** 이라는 같은 척도가 된다.

(호출 결과는 `analysis/data/naver_cache/`에 저장되어 다시 실행해도 API를 부르지 않습니다.)

In [ ]:
# 조회 기간: 가격 이력이 시작되는 시점 ~ 어제 (네이버는 어제까지 제공). 2024-01-01 이전으로는 가지 않습니다.
price_dates = (prices['collected_at'] + pd.Timedelta(hours=9)).dt.normalize()     # 저장은 UTC → 한국 날짜로 변환
START = min(pd.Timestamp('2024-01-01'), price_dates.min() - pd.Timedelta(days=30))
END = pd.Timestamp.now(tz='Asia/Seoul').normalize().tz_localize(None) - pd.Timedelta(days=1)
full_idx = pd.date_range(START, END, freq='D')
print('조회 기간:', START.date(), '~', END.date(), f'({len(full_idx)}일)')

# 네이버 호출 방식. 2026-07-31부터 신규 키는 NAVER API HUB(네이버 클라우드)에서만 발급됩니다. 주소와 인증 헤더가 개발자센터와 다릅니다.
NAVER_PLATFORM = 'apihub'          # 'apihub' = NAVER API HUB 키 / 'legacy' = developers.naver.com 에서 받은 옛 키
_PLATFORMS = {
    'apihub': {'url': 'https://naverapihub.apigw.ntruss.com/search-trend/v1/search',
               'id': 'X-NCP-APIGW-API-KEY-ID', 'secret': 'X-NCP-APIGW-API-KEY'},
    'legacy': {'url': 'https://openapi.naver.com/v1/datalab/search',
               'id': 'X-Naver-Client-Id', 'secret': 'X-Naver-Client-Secret'},
}
URL = _PLATFORMS[NAVER_PLATFORM]['url']

_CRED = {}
def naver_credentials():
    """키는 환경 변수 → 저장소의 .env(Git이 무시하는 파일) → 입력창 순서로 찾고, 한 번만 받아 재사용합니다. 출력하지 않습니다."""
    if not _CRED:
        cid, sec = os.environ.get('NAVER_CLIENT_ID'), os.environ.get('NAVER_CLIENT_SECRET')
        if not (cid and sec):
            for envf in (Path('../../.env'), Path('../.env'), Path('.env')):
                if envf.exists():
                    vals = {}
                    for line in envf.read_text(encoding='utf-8').splitlines():
                        line = line.strip()
                        if line and not line.startswith('#') and '=' in line:
                            k, v = line.split('=', 1)
                            vals[k.strip()] = v.strip().strip('"\'')
                    cid, sec = cid or vals.get('NAVER_CLIENT_ID'), sec or vals.get('NAVER_CLIENT_SECRET')
        cid = cid or getpass.getpass('NAVER_CLIENT_ID: ')
        sec = sec or getpass.getpass('NAVER_CLIENT_SECRET: ')
        _CRED.update({_PLATFORMS[NAVER_PLATFORM]['id']: cid, _PLATFORMS[NAVER_PLATFORM]['secret']: sec})
    return dict(_CRED)

def datalab(groups):
    # 같은 요청은 캐시에서 읽고, 없을 때만 API를 호출합니다.
    body = {'startDate': START.strftime('%Y-%m-%d'), 'endDate': END.strftime('%Y-%m-%d'),
            'timeUnit': 'date', 'keywordGroups': groups}
    payload = json.dumps(body, ensure_ascii=False, sort_keys=True)
    cache = CACHE_DIR / (hashlib.md5(payload.encode('utf-8')).hexdigest()[:16] + '.json')
    if cache.exists():
        return json.loads(cache.read_text(encoding='utf-8'))
    headers = {**naver_credentials(), 'Content-Type': 'application/json'}   # 키는 노트북에 저장되지 않음
    for attempt in range(5):
        r = requests.post(URL, headers=headers, data=payload.encode('utf-8'), timeout=30)
        if r.status_code == 200:
            cache.write_text(r.text, encoding='utf-8')
            time.sleep(0.2)
            return r.json()
        if r.status_code == 429:                       # 호출 한도 초과 → 잠깐 쉬고 재시도
            time.sleep(2 ** attempt)
            continue
        # 401: 키가 틀렸거나, 키 종류가 NAVER_PLATFORM 설정과 다르거나, Application에 '검색어 트렌드' API가 등록되지 않은 경우입니다.
        # 403: 주소가 https가 아니거나 필수 값이 빠졌을 때입니다.
        raise RuntimeError(f'데이터랩 호출 실패 {r.status_code}: {r.text[:200]}')
    raise RuntimeError('데이터랩 호출이 계속 한도 초과(429)입니다. 잠시 뒤 다시 실행하세요.')

def keywords_of(gid):
    return KEYWORD_OVERRIDE.get(gid) or [names[gid]]

anchor_group = {'groupName': ANCHOR_NAME, 'keywords': ANCHOR_KEYWORDS}
batches = [game_ids[i:i + 4] for i in range(0, len(game_ids), 4)]
print(f'요청 {len(batches)}번 필요 (기준 검색어 1 + 게임 4)')

In [ ]:
# 요청마다 호출하고, 기준 검색어 평균이 100이 되도록 척도를 맞춰 하나의 표로 합칩니다.
frames, anchor_means = [], []
for batch in batches:
    groups = [anchor_group] + [{'groupName': f'g{gid}', 'keywords': keywords_of(gid)} for gid in batch]
    res = datalab(groups)
    # 검색량이 없어 data가 빈 게임도 있으므로 dtype=float를 강제합니다(없으면 열이 object가 되어 이후 계산이 실패함)
    series = {r['title']: pd.Series({pd.Timestamp(x['period']): x['ratio'] for x in r['data']}, dtype=float) for r in res['results']}
    anchor = series[ANCHOR_NAME].reindex(full_idx).fillna(0).astype(float)       # 값이 없는 날은 검색이 없던 날로 봅니다
    scale = 100 / anchor.mean()
    anchor_means.append(anchor.mean())
    for gid in batch:
        raw = series[f'g{gid}'].reindex(full_idx).fillna(0).astype(float)
        frames.append(pd.DataFrame({'game_id': gid, 'date': full_idx, 'ratio_raw': raw.values, 'ratio': raw.values * scale}))

trend_long = pd.concat(frames, ignore_index=True)
trend = trend_long.pivot(index='date', columns='game_id', values='ratio').astype(float)       # 날짜 × 게임 표
print('검색량 표:', trend.shape, '(날짜 × 게임)')
print('요청별 기준 검색어 원값 평균: 최소 %.1f ~ 최대 %.1f  → 요청마다 척도가 다르므로 보정이 필요했습니다' % (min(anchor_means), max(anchor_means)))
trend.iloc[:5, :5]

## 4. 데이터 점검과 전처리
### 4-1) 구조·결측·기초 통계

In [ ]:
# 가격 이력과 게임 표의 구조·결측치를 확인합니다. 분석 전에 반드시 하는 점검입니다.
prices.info()

In [ ]:
# 컬럼별 결측치 개수. discount_pct가 비어 있으면 "할인 없음(0%)"으로 볼지 판단해야 합니다.
pd.DataFrame({'prices 결측': prices.isnull().sum(), 'prices 결측률(%)': prices.isnull().mean() * 100})

In [ ]:
# 결측 위치를 그림으로 확인 (노란 칸 = 결측)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
sns.heatmap(prices.sample(min(len(prices), 400), random_state=SEED).isnull(), cbar=False, yticklabels=False, cmap='viridis', ax=ax[0])
ax[0].set_title('가격 이력 결측 (무작위 400행)')
sns.heatmap(cand[['name_ko', 'release_date', 'review_count_ko', 'genres', 'is_online']].isnull(), cbar=False, yticklabels=False, cmap='viridis', ax=ax[1])
ax[1].set_title('분석 대상 게임 결측')
savefig(fig, 'a2_0_missing.png'); plt.show()

### 차트 해설
가격 이력(왼쪽)과 분석 대상 게임(오른쪽)의 빈 값 위치를 색으로 나타냅니다. 노란 칸이 결측입니다.

### 이 차트로 알 수 있는 것
- 어떤 컬럼에 결측이 몰려 있는지 → 삭제·대체 중 무엇이 필요한지 판단합니다. (예: `discount_pct`가 비면 0%로 대체, `release_date`가 비면 출시 화제성 분석에서 제외)
- 결측이 특정 게임에 몰려 있는지 → 몰려 있으면 그 게임은 분석에서 빼는 것이 안전합니다.

In [ ]:
# 숫자 컬럼의 기초 통계. 할인율이 0~100 범위인지, 가격이 음수·0 같은 이상값은 없는지 확인합니다.
prices[['discount_pct', 'price_krw', 'regular_krw']].describe()

In [ ]:
# 검색량 점검: 게임마다 평균 검색량(스팀=100)과 "검색이 0인 날의 비율"을 봅니다.
level = pd.DataFrame({'평균': trend.mean(), '0인날비율': (trend_long.pivot(index='date', columns='game_id', values='ratio_raw').astype(float) == 0).mean()})
level['name_ko'] = level.index.map(names)
low = level[(level['평균'] < MIN_LEVEL) | (level['0인날비율'] > 0.5)]
print(f'잡음이 큰 게임(평균 < {MIN_LEVEL} 또는 0인 날 50% 초과): {len(low)}개 → 분석에서 제외')
low.sort_values('평균').head(10)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(np.log10(level['평균'].clip(lower=1e-3)), bins=20, color=BLUE)
ax[0].axvline(np.log10(MIN_LEVEL), color=ORANGE, ls='--', label=f'제외 기준 {MIN_LEVEL}')
ax[0].set_xlabel('평균 검색량 (log10, 스팀=100 기준)'); ax[0].set_ylabel('게임 수'); ax[0].legend()
ax[0].set_title('게임별 평균 검색량 분포')
top = level.sort_values('평균', ascending=False).head(12)
ax[1].barh(top['name_ko'][::-1], top['평균'][::-1], color=BLUE); ax[1].set_title('검색량 상위 12개 게임')
ax[1].set_xlabel('평균 검색량 (스팀=100)')
savefig(fig, 'a2_1_level.png'); plt.show()

### 차트 해설
왼쪽은 게임별 평균 검색량의 분포(로그 눈금), 오른쪽은 검색량이 가장 큰 12개입니다. 모든 값이 "스팀 검색량 평균 = 100"이라는 같은 척도입니다.

### 이 차트로 알 수 있는 것
- 검색량이 거의 없는 게임(점선 왼쪽)은 하루 값이 0~1에서 출렁이는 잡음이라 변화율이 크게 왜곡되므로 분석에서 뺍니다.
- 검색량이 극단적으로 큰 게임이 있으면 그 게임이 결과를 좌우하지 않도록 이후 모든 계산을 **게임 안의 변화율(로그 배율)** 로 합니다.
- 오른쪽에 **이름이 일반 단어인 게임**(다른 의미의 검색이 섞였을 가능성)이 보이면 `EXCLUDE_IDS`에 넣거나 `KEYWORD_OVERRIDE`로 검색어를 바꾼 뒤 3-2부터 다시 실행합니다.

In [ ]:
# 잡음 게임을 제외합니다.
keep = level.index[(level['평균'] >= MIN_LEVEL) & (level['0인날비율'] <= 0.5)].tolist()
game_ids = [g for g in game_ids if g in keep]
trend = trend[game_ids]
print(f'분석에 쓰는 게임: {len(game_ids)}개')
R['n_games'] = len(game_ids)

### 4-1b) 요일 효과 보정

네이버 검색량은 **요일 리듬**이 있습니다(주말에 늘어나는 게임이 많음). 할인이 특정 요일에 몰리면, 할인 전후 곡선에 요일 효과가 섞여 보입니다.
게임마다 *요일별 평소 검색량 비율*을 구해 나눈 `trend_adj`를 만들어 두고, 5-3의 곡선과 7장 강건성 점검에서 원본과 비교합니다.
(6장 H1의 전후 **7일** 비교는 양쪽 모두 월~일을 한 번씩 포함해 요일이 균형이라 원본으로 검정합니다.)

In [ ]:
def deseasonalize(tr):
    # 게임마다 '요일별 평균 / 전체 평균'을 구해 그 비율로 나눕니다. (0이 되는 요일은 보정하지 않음)
    out = {}
    for gid in tr.columns:
        s = tr[gid]
        fac = (s.groupby(s.index.dayofweek).mean() / s.mean()).replace(0, np.nan).fillna(1.0)
        out[gid] = s / fac.reindex(s.index.dayofweek).to_numpy()
    return pd.DataFrame(out)

trend_adj = deseasonalize(trend)
print('요일 보정 검색량:', trend_adj.shape)

### 4-2) 가격 이력 → "일별 최고 할인율" (정제)

`price_history`는 **가격이 바뀐 시점만** 한 줄씩 쌓여 있고, 시각은 UTC입니다. 이것을 분석용 표로 바꾸는 과정:

1. 시각을 **한국 날짜**로 변환 (UTC +9시간) — 네이버 검색량이 한국 날짜 기준이기 때문
2. 스토어·에디션마다 *하루의 마지막 값*만 남기고, 변화가 없는 날은 **앞의 값을 이어 채움** (forward fill)
3. 같은 날 스토어 중 **가장 높은 할인율**을 그 게임의 그날 할인율로 봄 (소비자는 가장 싼 곳을 보기 때문)

In [ ]:
p = prices[prices['game_id'].isin(game_ids)].copy()
p['date'] = (p['collected_at'] + pd.Timedelta(hours=9)).dt.normalize()   # UTC → KST 날짜
p['discount_pct'] = p['discount_pct'].fillna(0)                          # 비어 있으면 할인 없음으로 처리

best_cols = {}
for gid, g in p.groupby('game_id'):
    daily = g.sort_values('collected_at').groupby(['store', 'edition', 'date'])['discount_pct'].last().unstack(['store', 'edition'])
    # 조회 시작일 이전의 마지막 값도 살리기 위해 두 날짜 축을 합쳐서 이어 채운 뒤 조회 기간만 자릅니다.
    daily = daily.reindex(daily.index.union(full_idx)).ffill().reindex(full_idx)
    best_cols[gid] = daily.max(axis=1)                                   # 같은 날 스토어 중 최고 할인율 (첫 기록 전은 NaN)
best = pd.DataFrame(best_cols)[game_ids]
print('일별 최고 할인율 표:', best.shape, '(날짜 × 게임)')
best.iloc[-5:, :5]

In [ ]:
# 정제가 맞게 됐는지 한 게임으로 눈으로 확인합니다: 원본 이력 몇 줄 vs 일별 최고 할인율.
ex = game_ids[0]
print('게임:', names[ex])
display(p[p['game_id'] == ex].sort_values('collected_at').head(6)[['store', 'edition', 'discount_pct', 'price_krw', 'collected_at']])
best[ex].dropna().loc[lambda s: s.diff().fillna(1) != 0].head(6).to_frame('일별 최고 할인율(변화가 있는 날만)')

### 4-3) 할인 시작 이벤트와 "비교 가능한" 위약 날짜 만들기

**할인 시작 이벤트** = 최고 할인율이 `MIN_DISC`(10%) 미만 → 이상으로 올라간 날. 다음 조건을 만족하는 것만 분석합니다.

| 조건 | 이유 |
| --- | --- |
| 시작 전 7일 내내 할인이 없었다 (깨끗한 비교 기간) | 이미 할인 중이던 게임의 연장은 "시작"이 아니다 |
| 시작 후 7일의 검색 자료가 있다 | 비교가 가능해야 한다 |
| 출시 후 30일 안이 아니다 | 출시 화제성과 섞인다 |

**위약(placebo) 날짜** = 같은 게임에서 *전후 7일 내내 할인이 없던* 임의의 날. 할인과 무관한 7일 변화가 얼마나 되는지 알려 줍니다.

**변화율 지표** = `log2( 시작 후 7일 평균 / 시작 전 7일 평균 )`. 0이면 변화 없음, 1이면 2배, −1이면 절반. 게임마다 검색량 크기가 달라도 같은 눈금으로 비교하고 극단값에 덜 흔들립니다. (보기 쉽게 `+N%`로도 환산)

In [ ]:
def eps_of(s):
    # 0으로 나누는 일을 막는 아주 작은 값: 그 게임의 "보통 검색량" 중앙값의 5%
    pos = s[s > 0]
    return 0.05 * (pos.median() if len(pos) else 1.0)

def build_samples(window=WINDOW, min_disc=MIN_DISC, seed=SEED, tr=None):
    # 게임마다 할인 시작 이벤트(event)와 위약 날짜(placebo)를 뽑아 전후 검색량 변화율을 계산합니다.
    tr = trend if tr is None else tr          # 기본은 원본 검색량, 요일 보정 표를 넘기면 그것으로 계산
    rng = np.random.default_rng(seed)
    rows, funnel = [], dict(시작일=0, 이전_할인없음=0, 출시_30일_밖=0, 자료_충분=0)
    for gid in game_ids:
        b, s = best[gid].to_numpy(), tr[gid].to_numpy()
        dates, n, eps = best.index, len(best), eps_of(tr[gid])
        rel = release.get(gid)

        def ok_release(i):
            if pd.isna(rel):
                return True
            gap = (dates[i] - rel).days
            return not (-window <= gap <= LAUNCH_EXCLUDE_DAYS)

        def calc(i):
            pre, post = s[i - window:i].mean(), s[i:i + window].mean()
            return pre, post, float(np.log2((post + eps) / (pre + eps)))

        # --- 할인 시작 이벤트 ---
        on = np.nan_to_num(b, nan=-1) >= min_disc
        prev_off = np.r_[False, (np.nan_to_num(b[:-1], nan=999) < min_disc)]
        for i in np.where(on & prev_off)[0]:
            funnel['시작일'] += 1
            if i < window or not np.all(np.nan_to_num(b[i - window:i], nan=999) < min_disc):
                continue
            funnel['이전_할인없음'] += 1
            if not ok_release(i):
                continue
            funnel['출시_30일_밖'] += 1
            if i + window > n:
                continue
            funnel['자료_충분'] += 1
            pre, post, l2 = calc(i)
            rows.append(dict(game_id=gid, date=dates[i], kind='event', depth=np.nanmax(b[i:i + 3]), pre=pre, post=post, log2=l2))

        # --- 위약 날짜: 전후 모두 할인 없는 날 ---
        cands = [i for i in range(window, n - window + 1)
                 if np.all(np.nan_to_num(b[i - window:i + window], nan=999) < min_disc) and ok_release(i)]
        if cands:
            for i in rng.choice(cands, size=min(PLACEBO_PER_GAME, len(cands)), replace=False):
                pre, post, l2 = calc(i)
                rows.append(dict(game_id=gid, date=dates[i], kind='placebo', depth=np.nan, pre=pre, post=post, log2=l2))
    df = pd.DataFrame(rows)
    df['pct'] = (2 ** df['log2'] - 1) * 100
    return df, funnel

samples, funnel = build_samples()
ev = samples[samples['kind'] == 'event'].copy()
pl = samples[samples['kind'] == 'placebo'].copy()
print(f'할인 시작 이벤트 {len(ev)}건 / 위약 날짜 {len(pl)}건 / 게임 {ev.game_id.nunique()}개에서 이벤트 발생')
R['n_events'], R['n_placebo'] = len(ev), len(pl)
samples.head()

In [ ]:
# 걸러지는 과정(깔때기): 어느 조건에서 이벤트가 많이 줄어드는지 봅니다.
fn = pd.Series(funnel)
fig, ax = plt.subplots(figsize=(7, 3.4))
bars = ax.barh(fn.index[::-1], fn.values[::-1], color=[GREEN, BLUE, BLUE, GRAY][::-1])
for bar, v in zip(bars, fn.values[::-1]):
    ax.text(v, bar.get_y() + bar.get_height() / 2, f' {v}', va='center')
ax.set_title('할인 시작 이벤트가 분석 표본이 되기까지'); ax.set_xlabel('이벤트 수')
savefig(fig, 'a2_2_funnel.png'); plt.show()
fn.to_frame('남은 이벤트 수')

### 차트 해설
할인 시작일로 잡힌 이벤트가 각 조건을 거치며 몇 건 남는지 보여 줍니다.

### 이 차트로 알 수 있는 것
- 표본이 어디서 줄었는지 투명하게 보여 줍니다. 특정 단계에서 대부분 사라진다면 그 조건이 결과에 영향을 줄 수 있어 7장 민감도 분석에서 다시 확인합니다.
- 최종 남은 수가 너무 작으면(예: 30건 미만) 아래 검정의 신뢰도가 낮다는 뜻입니다.

## 5. 탐색적 분석 (EDA)
### 5-1) 한 게임을 직접 보기 — 할인과 검색량이 같이 움직이나?

In [ ]:
# 이벤트가 가장 많은 게임 하나를 골라 검색량(선)과 할인율(계단)을 겹쳐 봅니다.
gid = ev['game_id'].value_counts().idxmax()
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(trend.index, trend[gid], color=BLUE, lw=1.2, label='검색량 (스팀=100)')
ax.set_ylabel('검색량'); ax.set_title(f'{names[gid]} — 검색량과 최고 할인율')
ax2 = ax.twinx()
ax2.step(best.index, best[gid], where='post', color=ORANGE, alpha=.8, label='최고 할인율(%)')
ax2.set_ylabel('할인율(%)'); ax2.grid(False)
for d in ev.loc[ev['game_id'] == gid, 'date']:
    ax.axvline(d, color=GREEN, ls=':', lw=1)
ax.plot([], [], color=GREEN, ls=':', label='할인 시작 이벤트')
h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, loc='upper left')
savefig(fig, 'a2_3_example.png'); plt.show()

### 차트 해설
한 게임의 검색량(파란 선)과 최고 할인율(주황 계단)을 같은 시간축에 겹친 그림입니다. 초록 점선이 할인 시작 이벤트입니다.

### 이 차트로 알 수 있는 것
- 할인 시작 직후 검색량이 솟는 모양이 보이는지, 아니면 할인과 상관없이 출렁이는지 **눈으로 먼저 확인**합니다. (한 게임의 사례이므로 결론이 아니라 가설의 힌트일 뿐입니다.)
- 출시 직후의 큰 봉우리처럼 할인과 무관한 큰 사건이 섞여 있는지 확인합니다. 이런 구간은 분석에서 제외한 부분입니다.

### 5-2) 이벤트·위약의 분포

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.8))
ax[0].hist(ev['depth'], bins=range(10, 101, 5), color=BLUE); ax[0].set_title('할인 시작 이벤트의 할인율 분포'); ax[0].set_xlabel('할인율(%)'); ax[0].set_ylabel('이벤트 수')
per_game = ev['game_id'].value_counts()
ax[1].hist(per_game.values, bins=range(1, per_game.max() + 2), color=BLUE); ax[1].set_title('게임당 이벤트 수'); ax[1].set_xlabel('이벤트 수'); ax[1].set_ylabel('게임 수')
ax[2].scatter(ev['pre'], ev['post'], s=14, alpha=.5, color=BLUE, label='할인 시작')
ax[2].scatter(pl['pre'], pl['post'], s=14, alpha=.35, color=GRAY, label='위약(할인 없음)')
lim = [max(1e-2, min(samples['pre'].min(), samples['post'].min())), max(samples['pre'].max(), samples['post'].max())]
ax[2].plot(lim, lim, color='k', lw=.8, ls='--'); ax[2].set_xscale('log'); ax[2].set_yscale('log'); plain_log(ax[2], 'x'); plain_log(ax[2], 'y'); ax[2].legend()
ax[2].set_xlabel('시작 전 7일 평균'); ax[2].set_ylabel('시작 후 7일 평균'); ax[2].set_title('전후 평균 비교 (점선 위 = 늘어남)')
savefig(fig, 'a2_4_distribution.png'); plt.show()

### 차트 해설
왼쪽은 할인 시작 이벤트가 어떤 할인율로 몰려 있는지, 가운데는 게임마다 이벤트가 몇 번인지, 오른쪽은 전후 7일 평균 검색량을 점으로 찍은 것입니다 (대각선 위쪽 = 할인 후 검색이 늘어난 경우).

### 이 차트로 알 수 있는 것
- 왼쪽: 할인율 구간별 표본 수. **표본이 적은 구간은 6장 H2에서 해석을 조심**해야 합니다.
- 가운데: 소수 게임이 이벤트를 독차지하면 결과가 그 게임들 때문일 수 있어, 신뢰구간을 게임 단위로 계산하는 이유가 됩니다.
- 오른쪽: 파란 점(할인)이 회색 점(위약)보다 대각선 위쪽에 더 많이 몰려 있는지 대략 가늠합니다. 정확한 판정은 6장의 검정이 합니다.

### 5-2b) 할인은 어느 요일에 시작되나

요일 효과가 문제가 되는지 먼저 확인합니다. 할인 시작 요일이 위약(무작위 날)과 다르게 한쪽에 몰려 있으면, 곡선의 "정점" 같은 모양이 **요일 효과**일 수 있습니다.

In [ ]:
DOW = ['월', '화', '수', '목', '금', '토', '일']
dow_tab = pd.DataFrame({'할인 시작': ev['date'].dt.dayofweek.value_counts().reindex(range(7), fill_value=0),
                        '위약': pl['date'].dt.dayofweek.value_counts().reindex(range(7), fill_value=0)})
dow_tab.index = DOW
chi = stats.chi2_contingency(dow_tab.to_numpy().T)
R['dow'] = dict(chi2_p=float(chi[1]), top_day=DOW[int(dow_tab['할인 시작'].to_numpy().argmax())],
                top_share=float(dow_tab['할인 시작'].max() / dow_tab['할인 시작'].sum()))
display(dow_tab.T)

fig, ax = plt.subplots(figsize=(7.5, 3.6))
share, x_ = dow_tab / dow_tab.sum(), np.arange(7)
ax.bar(x_ - 0.2, share['할인 시작'], 0.4, color=BLUE, label=f"할인 시작 (n={int(dow_tab['할인 시작'].sum())})")
ax.bar(x_ + 0.2, share['위약'], 0.4, color=GRAY, label=f"위약 (n={int(dow_tab['위약'].sum())})")
ax.set_xticks(x_); ax.set_xticklabels(DOW); ax.set_ylabel('비율'); ax.legend()
ax.set_title(f"시작 요일 분포 (할인 vs 위약 차이 검정 p={chi[1]:.1e})")
savefig(fig, 'a2_4b_weekday.png'); plt.show()
print(f"할인 시작의 {R['dow']['top_share']:.0%}가 {R['dow']['top_day']}요일입니다. 위약(무작위 날)과 요일 분포가 "
      + ('**다릅니다** → 곡선의 요일 효과를 보정해서 봐야 합니다.' if chi[1] < ALPHA else '크게 다르지 않습니다.'))

### 차트 해설
막대는 요일별 비율입니다. 파랑이 실제 할인 시작, 회색이 무작위로 뽑은 "할인 없는 날"(위약)입니다. 위약은 요일이 고르게 퍼지는 것이 정상입니다.

### 이 차트로 알 수 있는 것
- 파란 막대가 특정 요일에 높게 솟아 있으면 **할인이 그 요일에 몰려 시작**된다는 뜻입니다(예: 스토어의 주간 할인 일정). 이때 시작 2일 뒤(주말)의 검색 증가를 할인 효과로 읽으면 **착시**입니다.
- 그래서 5-3의 곡선은 원본과 요일 보정본을 나란히 그립니다.

### 5-3) 이벤트 정렬 평균 곡선 (event study)

모든 할인 시작일을 **0일**로 맞추고, 시작 전 7일 평균을 100으로 놓았을 때 검색량이 날마다 어떻게 변하는지 중앙값으로 봅니다. 위약 날짜도 같은 방식으로 그려 비교합니다. **왼쪽은 원본, 오른쪽은 4-1b의 요일 보정 검색량**입니다.

In [ ]:
PRE, POST = 14, 14

def curve(df, tr=None):
    # 각 날짜(0일 기준 -14 ~ +13)의 검색량을 "시작 전 7일 평균 = 100"으로 환산해 모읍니다.
    tr = trend if tr is None else tr
    out = []
    for _, r in df.iterrows():
        s = tr[r['game_id']]
        i = s.index.get_loc(r['date'])
        if i - PRE < 0 or i + POST > len(s):
            continue
        seg = s.iloc[i - PRE:i + POST].to_numpy()
        base = s.iloc[i - WINDOW:i].mean() + eps_of(s)
        row = {'game_id': r['game_id']}
        row.update({d: v for d, v in zip(range(-PRE, POST), seg / base * 100)})    # 열 이름 = 0일 기준 경과일(정수)
        out.append(row)
    return pd.DataFrame(out)

days = list(range(-PRE, POST))

def band(cv, B=500, seed=SEED):
    # 게임 단위로 다시 뽑아(cluster bootstrap) 중앙값 곡선의 95% 신뢰대를 만듭니다.
    rng = np.random.default_rng(seed)
    groups = {g: v[days].to_numpy() for g, v in cv.groupby('game_id')}
    keys = np.array(list(groups))
    meds = np.array([np.median(np.vstack([groups[k] for k in rng.choice(keys, len(keys))]), axis=0) for _ in range(B)])
    return np.percentile(meds, [2.5, 97.5], axis=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.4), sharey=True)
curves = {}
for ax, (title, tr) in zip(axes, [('원본 검색량', trend), ('요일 보정 검색량', trend_adj)]):
    cv_e, cv_p = curve(ev, tr), curve(pl, tr)
    curves[title] = (cv_e, cv_p)
    for cv, col, lab in [(cv_p, GRAY, '위약(할인 없음)'), (cv_e, BLUE, '할인 시작')]:
        lo, hi = band(cv)
        ax.plot(days, cv[days].median(), color=col, lw=2, label=f'{lab} (n={len(cv)})')
        ax.fill_between(days, lo, hi, color=col, alpha=.2)
    ax.axvline(0, color=ORANGE, ls='--'); ax.axhline(100, color='k', lw=.6)
    ax.set_xlabel('할인 시작일 기준 경과일'); ax.set_title(title); ax.legend()
axes[0].set_ylabel('검색량 지수 (시작 전 7일 평균 = 100)')
fig.suptitle('할인 시작 전후 검색량 (중앙값, 음영 = 95% 신뢰구간)', y=1.02)
savefig(fig, 'a2_5_event_study.png'); plt.show()
cv_ev, cv_pl = curves['원본 검색량']

# 활용: 광고 노출 시점의 근거. 시작 전 7일 중앙값 수준 대비 첫 주(1~7일)·둘째 주(8~13일) 평균이 얼마나 높은지, 뚜렷한 정점이 있는지
def peak_stats(cv):
    med = cv[days].median(); after = med.loc[0:]
    pre_level = med.loc[-WINDOW:-1].mean()
    w1, w2 = med.loc[1:7].mean(), med.loc[8:].mean()
    peak_day, peak_idx = int(after.idxmax()), float(after.max())
    return dict(peak_day=peak_day, peak_index=peak_idx, pre_level=float(pre_level),
                week1_pct=float((w1 / pre_level - 1) * 100), week2_pct=float((w2 / pre_level - 1) * 100),
                peak_clear=bool(peak_idx - w1 >= PEAK_MARGIN))

R['peak_raw'] = peak_stats(curves['원본 검색량'][0]); R['peak_adj'] = peak_stats(curves['요일 보정 검색량'][0])
R['peak_day'], R['peak_index'] = R['peak_adj']['peak_day'], R['peak_adj']['peak_index']
for nm, key in (('원본', 'peak_raw'), ('요일 보정', 'peak_adj')):
    s_ = R[key]
    print(f"[{nm}] 시작 후 가장 높은 날 {s_['peak_day']}일차(지수 {s_['peak_index']:.0f}) / 첫 주 평균 {s_['week1_pct']:+.0f}%, "
          f"둘째 주 평균 {s_['week2_pct']:+.0f}% (시작 전 7일 수준 대비) / 뚜렷한 정점: {'있음' if s_['peak_clear'] else '없음'}")

### 차트 해설
가로축은 할인 시작일(0)을 기준으로 한 경과일, 세로축은 **시작 전 7일 평균을 100으로 놓은 검색량 지수**입니다. 파란 선이 할인 시작 이벤트, 회색 선이 할인이 없는 보통 날(위약)이며, 음영은 95% 신뢰구간입니다. **왼쪽은 원본, 오른쪽은 요일 효과를 보정한 검색량**입니다.

### 이 차트로 알 수 있는 것
- 0일을 지나며 파란 선이 100 위로 올라가고 회색 선과 음영이 **겹치지 않으면**, 할인 시작과 함께 관심이 늘었다는 직접적인 근거입니다.
- **두 패널을 비교**하세요. 원본에서 보이는 뚜렷한 정점이 오른쪽(요일 보정)에서 사라지면 그 정점은 **요일 효과**입니다. 정점의 간격이 정확히 7일이면 요일 효과를 의심합니다.
- 광고·프로모션을 *언제 집행할지*의 근거는 요일 보정본의 첫 주·둘째 주 평균과 정점 유무(위 셀의 출력)로 판단합니다.
- 시작 전(−14~−1일)에 이미 오르고 있으면 할인 발표·시즌 이벤트 등 다른 원인이 섞였을 수 있습니다.

## 6. 가설 검정

공통 도구를 먼저 정의합니다. (효과크기 Cliff's delta, 게임 단위 부트스트랩 신뢰구간, 판정 함수)

In [ ]:
def cliffs_delta(x, y):
    # 두 집단에서 무작위로 하나씩 뽑았을 때 x가 y보다 클 확률 - 작을 확률. -1~1, 0이면 차이 없음.
    u = stats.mannwhitneyu(x, y, alternative='two-sided').statistic
    return 2 * u / (len(x) * len(y)) - 1

def cluster_ci(df, col='log2', stat=np.median, B=N_BOOT, seed=SEED):
    # 같은 게임의 이벤트는 비슷하게 움직이므로, 게임을 단위로 다시 뽑아 신뢰구간을 구합니다.
    groups = {g: v[col].to_numpy() for g, v in df.groupby('game_id')}
    keys = np.array(list(groups))
    rng = np.random.default_rng(seed)
    vals = [stat(np.concatenate([groups[k] for k in rng.choice(keys, len(keys))])) for _ in range(B)]
    return np.percentile(vals, [2.5, 97.5])

def to_pct(l2):
    return (2 ** l2 - 1) * 100

def verdict(p, effect_ok, direction_ok=True):
    if direction_ok and p < ALPHA and effect_ok:
        return '✅ 가설과 일치 — 통계적으로 유의하고 효과도 실무 기준을 넘음'
    if direction_ok and p < ALPHA:
        return '⚠️ 부분 일치 — 통계적으로는 유의하나 효과가 실무 기준(+20%)에 못 미침'
    return '❌ 불일치 — 귀무가설을 기각하지 못함(차이가 있다고 말할 근거 부족)'

### 6-1) H1. 할인이 시작되면 한국 검색 관심이 늘어나는가?

**① 정규성 확인 → 검정 도구 선택**: 변화율이 정규분포가 아니면 t-검정 대신 순위 기반 **Wilcoxon 부호순위검정**(전후 짝지은 자료, 단측)을 씁니다.
**② 위약 대조**: 할인 없는 날의 변화와 **Mann-Whitney U**로 비교합니다. (할인이 "원래 있는 출렁임"보다 큰가)

In [ ]:
x, y = ev['log2'].to_numpy(), pl['log2'].to_numpy()

# ① 정규성
sw = stats.shapiro(x[:5000])
print(f'Shapiro-Wilk 정규성 검정: W={sw.statistic:.3f}, p={sw.pvalue:.4f}  →',
      '정규분포가 아님 → 비모수 검정 사용' if sw.pvalue < ALPHA else '정규성을 부정할 근거 없음(그래도 비모수 검정으로 보수적으로 확인)')

# 할인 전후 짝지은 비교 (Wilcoxon 부호순위, 단측: 후 > 전)
w = stats.wilcoxon(ev['post'], ev['pre'], alternative='greater')
# ② 위약 대조 (Mann-Whitney U, 단측: 할인 > 위약)
mw = stats.mannwhitneyu(x, y, alternative='greater')
delta = cliffs_delta(x, y)
lo, hi = cluster_ci(ev)
med_ev, med_pl = np.median(x), np.median(y)

h1 = pd.DataFrame({
    '값': [len(x), len(y), f'{to_pct(med_ev):+.1f}%', f'{to_pct(med_pl):+.1f}%', f'{to_pct(lo):+.1f}% ~ {to_pct(hi):+.1f}%',
           f'{w.pvalue:.2e}', f'{mw.pvalue:.2e}', f'{delta:.2f}']},
    index=['할인 이벤트 수', '위약 수', '검색량 변화 중앙값 (할인)', '검색량 변화 중앙값 (위약)', '할인 중앙값 95% 신뢰구간(게임 단위)',
           'Wilcoxon p (후 > 전)', 'Mann-Whitney p (할인 > 위약)', "Cliff's delta (할인 vs 위약)"])
display(h1)

effect_ok = med_ev >= PRACTICAL_LOG2 and lo > 0
R['h1'] = dict(n=len(x), median_pct=float(to_pct(med_ev)), placebo_median_pct=float(to_pct(med_pl)),
               ci_pct=[float(to_pct(lo)), float(to_pct(hi))], p_wilcoxon=float(w.pvalue), p_mwu=float(mw.pvalue),
               cliffs_delta=float(delta), ok=bool(mw.pvalue < ALPHA and w.pvalue < ALPHA and effect_ok))
sig_h1 = bool(mw.pvalue < ALPHA and w.pvalue < ALPHA and med_ev > med_pl)
R['h1']['status'] = 'match' if (sig_h1 and effect_ok) else ('partial' if sig_h1 else 'mismatch')
print('\nH1 판정:', verdict(max(w.pvalue, mw.pvalue), effect_ok, direction_ok=med_ev > med_pl))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
bins = np.linspace(-2, 3, 41)
ax[0].hist(np.clip(y, -2, 3), bins=bins, alpha=.6, color=GRAY, density=True, label=f'위약 (중앙값 {to_pct(med_pl):+.0f}%)')
ax[0].hist(np.clip(x, -2, 3), bins=bins, alpha=.6, color=BLUE, density=True, label=f'할인 시작 (중앙값 {to_pct(med_ev):+.0f}%)')
ax[0].axvline(0, color='k', lw=.8); ax[0].axvline(PRACTICAL_LOG2, color=ORANGE, ls='--', label='실무 기준 +20%')
ax[0].set_xlabel('검색량 변화율 log2(후/전)  (0=변화없음, 1=2배)'); ax[0].set_ylabel('밀도'); ax[0].legend(); ax[0].set_title('변화율 분포 (범위를 벗어난 값은 가장자리로 모음)')
data = pd.concat([ev.assign(구분='할인 시작'), pl.assign(구분='위약')])
sns.boxplot(data=data, x='구분', y='log2', palette=[BLUE, GRAY], showfliers=False, ax=ax[1], order=['할인 시작', '위약'])
sns.stripplot(data=data.sample(min(len(data), 400), random_state=SEED), x='구분', y='log2', color='k', alpha=.25, size=3, ax=ax[1], order=['할인 시작', '위약'])
ax[1].axhline(0, color='k', lw=.8); ax[1].axhline(PRACTICAL_LOG2, color=ORANGE, ls='--'); ax[1].set_ylabel('log2(후/전)'); ax[1].set_title('중앙값 비교')
savefig(fig, 'a2_6_h1.png'); plt.show()

### 차트 해설
왼쪽은 변화율 분포(파랑 = 할인 시작, 회색 = 위약), 오른쪽은 상자그림입니다. 점선(주황)은 "의미 있다"의 실무 기준(+20%)입니다.

### 이 차트로 알 수 있는 것
- 파란 분포가 회색보다 **오른쪽으로 치우쳐 있고** 중앙값이 점선을 넘으면 H1과 일치합니다.
- 두 분포가 거의 겹치면, 할인이 있든 없든 비슷하게 출렁인다는 뜻입니다.
- 위의 표에서 **p값은 "우연일 확률", 효과크기(중앙값 변화·Cliff's delta)는 "얼마나 큰 차이인가"** 입니다. 둘 다 봐야 의미 있는 결과입니다.

### 6-2) H2. 할인율이 클수록 관심이 더 늘고, 효과가 나는 최소 할인율이 있는가?

* **Spearman 상관**: 할인율과 변화율이 같이 커지는가 (순서 기반이라 이상치에 강함)
* **Kruskal-Wallis**: 할인율 구간 사이에 변화율 차이가 있는가 (3개 이상 집단 비모수 비교)
* **구간별 Wilcoxon (+ Holm 보정)**: 어느 구간부터 0보다 유의하게 큰가. 구간 4개를 동시에 검정하므로 *여러 번 검정하면 우연히 유의해지는 문제*를 Holm 방법으로 보정합니다.
* **구간별 위약 비교(Mann-Whitney, Holm 보정)**: 보통 날보다 큰가

In [ ]:
ev['bin'] = pd.cut(ev['depth'], bins=DISC_BINS, labels=DISC_LABELS, right=False)
MIN_BIN_N = 8   # 구간 표본이 이보다 적으면 검정하지 않고 "표본 부족"으로 표시

rho, p_rho = stats.spearmanr(ev['depth'], ev['log2'])
groups = [g['log2'].to_numpy() for _, g in ev.groupby('bin', observed=True) if len(g) >= MIN_BIN_N]
kw = stats.kruskal(*groups) if len(groups) >= 2 else None
print(f"Spearman 상관(할인율↔변화율): rho={rho:.3f}, p={p_rho:.4f}")
print(f"Kruskal-Wallis(구간 간 차이): H={kw.statistic:.2f}, p={kw.pvalue:.4f}" if kw else 'Kruskal-Wallis: 표본이 충분한 구간이 2개 미만')

rows = []
for lab in DISC_LABELS:
    g = ev.loc[ev['bin'] == lab, 'log2'].to_numpy()
    if len(g) < MIN_BIN_N:
        rows.append(dict(구간=lab, n=len(g), 중앙값_변화=np.nan, 신뢰구간='표본 부족', p_0보다큼=np.nan, p_위약보다큼=np.nan, 효과크기=np.nan))
        continue
    sub = ev[ev['bin'] == lab]
    lo_, hi_ = cluster_ci(sub, B=1000)
    rows.append(dict(구간=lab, n=len(g), 중앙값_변화=to_pct(np.median(g)), 신뢰구간=f'{to_pct(lo_):+.0f}% ~ {to_pct(hi_):+.0f}%',
                     p_0보다큼=stats.wilcoxon(g, alternative='greater').pvalue if np.any(g != 0) else 1.0,
                     p_위약보다큼=stats.mannwhitneyu(g, y, alternative='greater').pvalue, 효과크기=cliffs_delta(g, y)))
h2 = pd.DataFrame(rows)
tested = h2['p_0보다큼'].notna()
for col in ['p_0보다큼', 'p_위약보다큼']:                       # Holm 보정
    h2.loc[tested, col + '(보정)'] = multipletests(h2.loc[tested, col], method='holm')[1]
h2['유의(둘 다)'] = (h2['p_0보다큼(보정)'] < ALPHA) & (h2['p_위약보다큼(보정)'] < ALPHA)
h2['실무기준 통과'] = h2['중앙값_변화'] >= (PRACTICAL - 1) * 100
h2['의미 있음'] = h2['유의(둘 다)'] & h2['실무기준 통과']
display(h2.round(4))

meaningful = h2.loc[h2['의미 있음'].fillna(False), '구간']
R['h2'] = dict(spearman_rho=float(rho), spearman_p=float(p_rho), kruskal_p=float(kw.pvalue) if kw else None,
               meaningful_bins=meaningful.tolist(), min_bin=meaningful.iloc[0] if len(meaningful) else None,
               ok=bool(p_rho < ALPHA and rho > 0 and len(meaningful) > 0),
               table=h2[['구간', 'n', '중앙값_변화']].fillna(-999).to_dict('records'))
R['h2']['n_sig_bins'] = int(h2['유의(둘 다)'].fillna(False).sum())      # 위약과 0보다 모두 유의하게 큰 구간 수
R['h2']['n_tested_bins'] = int(tested.sum())
R['h2']['status'] = 'match' if R['h2']['ok'] else ('partial' if (p_rho < ALPHA and rho > 0) else 'mismatch')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
sns.boxplot(data=ev, x='bin', y='log2', color=BLUE, showfliers=False, ax=ax[0], order=DISC_LABELS)
sns.stripplot(data=ev, x='bin', y='log2', color='k', alpha=.25, size=3, ax=ax[0], order=DISC_LABELS)
ax[0].axhline(0, color='k', lw=.8); ax[0].axhline(np.median(y), color=GRAY, ls='--', label='위약 중앙값')
ax[0].axhline(PRACTICAL_LOG2, color=ORANGE, ls='--', label='실무 기준 +20%')
ax[0].set_xticks(range(len(DISC_LABELS)))
ax[0].set_xticklabels([f"{lab}\n(n={int((ev['bin'] == lab).sum())})" for lab in DISC_LABELS])
ax[0].set_xlabel('할인율 구간'); ax[0].set_ylabel('검색량 변화율 log2(후/전)'); ax[0].legend(loc='lower right'); ax[0].set_title('할인율 구간별 검색량 변화')

# "할인율이 t% 이상인 이벤트 전체"의 효과: t를 올려 가며 위약과 비교
thr = list(range(10, 80, 10)); trows = []
for t in thr:
    g = ev.loc[ev['depth'] >= t, 'log2'].to_numpy()
    if len(g) >= MIN_BIN_N:
        trows.append((t, len(g), to_pct(np.median(g)), stats.mannwhitneyu(g, y, alternative='greater').pvalue))
tr = pd.DataFrame(trows, columns=['t', 'n', 'pct', 'p'])
ax[1].plot(tr['t'], tr['pct'], marker='o', color=BLUE)
for _, r in tr.iterrows():
    ax[1].annotate(f"n={int(r['n'])}\np={r['p']:.3f}", (r['t'], r['pct']), textcoords='offset points', xytext=(0, 8), ha='center', fontsize=8)
ax[1].axhline((PRACTICAL - 1) * 100, color=ORANGE, ls='--', label='실무 기준 +20%'); ax[1].axhline(0, color='k', lw=.8)
ax[1].set_xlabel('할인율이 이 값 이상인 이벤트'); ax[1].set_ylabel('검색량 변화 중앙값(%)'); ax[1].legend(); ax[1].set_title('할인율 기준을 올릴수록 효과가 커지나?')
savefig(fig, 'a2_7_h2.png'); plt.show()

### 차트 해설
왼쪽은 할인율 구간별 검색량 변화 분포(위쪽 숫자는 표본 수), 오른쪽은 "할인율이 이 값 **이상**인 이벤트 전체"의 변화 중앙값입니다.

### 이 차트로 알 수 있는 것
- 구간이 올라갈수록 상자가 위로 올라가는지(계단식 증가), 아니면 어느 지점에서 갑자기 올라가는지(문턱 효과), 아니면 평평한지 확인합니다.
- 상자가 **회색 점선(위약)과 주황 점선(실무 기준)을 모두 넘는 가장 낮은 구간**이 "퍼블리셔에게 제시할 최소 할인율" 후보입니다.
- 위 표의 **표본 부족** 구간은 결론을 내지 않습니다. 오른쪽 그림의 `n`이 작을수록 해당 점의 신뢰도가 낮습니다.

### 6-3) H3. 출시 화제성은 3주 안에 절반으로 식는가?

**방법**
1. 출시일 앞뒤로 검색량을 7일 이동평균으로 다듬는다. (요일 효과 제거)
2. 출시 후 30일 안의 **최고점(정점)** 을 찾고, 그 값의 **절반 이하로 처음 내려가는 날**까지의 일수를 센다. (출시일 기준)
3. 관측 기간(출시 후 90일 또는 조회 종료일) 안에 절반이 안 된 게임은 **중도절단(censored)** 으로 두고 버리지 않는다 → **Kaplan-Meier**로 생존곡선·중앙값을 구한다.
4. 검정: 출시 후 21일 안에 절반이 된 게임의 비율이 50%보다 큰가 → **이항검정(단측)** . (중앙값 ≤ 21일과 같은 말)

출시 화제성이 뚜렷하지 않은 게임(정점이 평소 검색량의 2배 미만)은 *반감기를 정의할 수 없어* 제외하고 개수를 밝힙니다.

In [ ]:
OBS_DAYS, PEAK_DAYS = 90, 30
rows = []
for gid in game_ids:
    rel = release.get(gid)
    if pd.isna(rel) or rel < START + pd.Timedelta(days=7) or rel > END - pd.Timedelta(days=OBS_DAYS // 2):
        continue
    s = trend[gid].rolling(7, center=True, min_periods=4).mean()
    seg = s.loc[rel:rel + pd.Timedelta(days=OBS_DAYS)]
    if len(seg) < 45:
        continue
    peak_seg = seg.iloc[:PEAK_DAYS + 1]
    peak_day, peak = (peak_seg.idxmax() - rel).days, peak_seg.max()
    base = trend[gid].median()
    if peak < 2 * max(base, 1e-9):                          # 평소의 2배도 안 되면 "화제성"이 없다고 보고 제외
        rows.append(dict(game_id=gid, 상태='제외(화제성 뚜렷하지 않음)', 정점일=peak_day, 정점=peak, 반감일수=np.nan, 관측=np.nan)); continue
    after = seg.loc[peak_seg.idxmax():]
    below = after[after <= peak / 2]
    if len(below):
        rows.append(dict(game_id=gid, 상태='분석', 정점일=peak_day, 정점=peak, 반감일수=(below.index[0] - rel).days, 관측=1))
    else:
        rows.append(dict(game_id=gid, 상태='분석', 정점일=peak_day, 정점=peak, 반감일수=(seg.index[-1] - rel).days, 관측=0))   # 중도절단
dec = pd.DataFrame(rows)
dec['name_ko'] = dec['game_id'].map(names)
print('출시 후 90일 관측이 가능한 게임:', len(dec), '개')
print(dec['상태'].value_counts().to_string())
used = dec[dec['상태'] == '분석'].copy()
dec.head(8)

In [ ]:
def kaplan_meier(t, e):
    # t: 반감까지 일수(또는 관측 종료까지), e: 1=반감 관측, 0=중도절단. 생존함수 S(t) = "아직 절반이 안 된 비율"
    order = np.argsort(t); t, e = np.asarray(t)[order], np.asarray(e)[order]
    s, out, at_risk = 1.0, [(0, 1.0)], len(t)
    for tt in np.unique(t):
        d = int(((t == tt) & (e == 1)).sum()); n_t = int((t >= tt).sum())
        if d: s *= 1 - d / n_t
        out.append((tt, s))
    return pd.DataFrame(out, columns=['day', 'S'])

if len(used) >= 5:
    km = kaplan_meier(used['반감일수'], used['관측'])
    median_day = km.loc[km['S'] <= 0.5, 'day'].min()
    k = int(((used['관측'] == 1) & (used['반감일수'] <= DECAY_DAYS)).sum())
    bt = stats.binomtest(k, len(used), 0.5, alternative='greater')
    print(f'분석 게임 {len(used)}개 중 {DECAY_DAYS}일 안에 절반이 된 게임: {k}개 ({k / len(used):.0%}), 중도절단 {int((used["관측"] == 0).sum())}개')
    print('Kaplan-Meier 반감 중앙값:', f'{median_day:.0f}일' if pd.notna(median_day) else '관측 기간 안에서는 50%가 반감에 도달하지 못함')
    print(f'이항검정(21일 안에 반감한 비율 > 50%): p={bt.pvalue:.4f}')
    h3_ok = (pd.notna(median_day) and median_day <= DECAY_DAYS) and bt.pvalue < ALPHA
    R['h3'] = dict(n=len(used), k_within=k, share_within=k / len(used), km_median_days=None if pd.isna(median_day) else float(median_day),
                   p_binom=float(bt.pvalue), censored=int((used['관측'] == 0).sum()), excluded=int((dec['상태'] != '분석').sum()), ok=bool(h3_ok))
    print('\nH3 판정:', verdict(bt.pvalue, effect_ok=(pd.notna(median_day) and median_day <= DECAY_DAYS)))
    R['h3']['status'] = 'match' if h3_ok else 'mismatch'
else:
    km = None; h3_ok = None
    print('출시 화제성을 분석할 수 있는 게임이 5개 미만이라 H3는 검정하지 않습니다. (가격 이력이 ~23개월이라 최근 출시작이 적을 수 있음)')
    R['h3'] = None

In [ ]:
if km is not None:
    fig, ax = plt.subplots(1, 3, figsize=(15, 4))
    # (1) 게임별 출시 후 검색량 곡선 (정점=100)
    for _, r in used.iterrows():
        rel = release[r['game_id']]
        s = trend[r['game_id']].rolling(7, center=True, min_periods=4).mean().loc[rel:rel + pd.Timedelta(days=OBS_DAYS)]
        ax[0].plot((s.index - rel).days, s / s.iloc[:PEAK_DAYS + 1].max() * 100, color=BLUE, alpha=.25, lw=1)
    ax[0].axhline(50, color=ORANGE, ls='--', label='정점의 절반'); ax[0].axvline(DECAY_DAYS, color=GRAY, ls=':', label=f'{DECAY_DAYS}일')
    ax[0].set_xlabel('출시 후 경과일'); ax[0].set_ylabel('검색량 (출시 후 정점 = 100)'); ax[0].set_title('게임별 출시 화제성 곡선'); ax[0].legend()
    # (2) 반감까지 일수 분포
    ax[1].hist(used.loc[used['관측'] == 1, '반감일수'], bins=15, color=BLUE)
    ax[1].axvline(DECAY_DAYS, color=ORANGE, ls='--', label=f'{DECAY_DAYS}일'); ax[1].legend()
    ax[1].set_xlabel('출시 후 절반이 되기까지 일수 (관측된 게임)'); ax[1].set_ylabel('게임 수'); ax[1].set_title('반감까지 걸린 기간')
    # (3) Kaplan-Meier
    ax[2].step(km['day'], km['S'], where='post', color=BLUE, lw=2)
    ax[2].axhline(0.5, color=ORANGE, ls='--'); ax[2].axvline(DECAY_DAYS, color=GRAY, ls=':')
    ax[2].set_xlabel('출시 후 경과일'); ax[2].set_ylabel('아직 절반이 안 된 게임 비율'); ax[2].set_title('Kaplan-Meier 생존곡선'); ax[2].set_ylim(0, 1.02)
    savefig(fig, 'a2_8_h3.png'); plt.show()

### 차트 해설
왼쪽은 게임별 출시 후 검색량 곡선(출시 후 정점을 100으로 맞춤), 가운데는 절반이 되기까지 걸린 일수의 분포, 오른쪽은 Kaplan-Meier 생존곡선입니다. 생존곡선이 0.5(주황 가로선)를 지나는 날이 **반감 중앙값**입니다.

### 이 차트로 알 수 있는 것
- 왼쪽에서 곡선이 점선(21일)에 오기 전에 주황선(50)을 대부분 지나가면 "3주면 절반" 가설과 일치합니다.
- 곡선이 아예 내려오지 않는 게임이 많다면(중도절단) 출시 후에도 꾸준히 검색되는 게임이 많다는 뜻이고, 그만큼 "빨리 식는다"는 가설은 약해집니다.
- 이 분석은 *가격 이력이 시작된 뒤에 출시한 게임*만 쓸 수 있어 표본이 작습니다. 표본 수(n)와 제외 수를 같이 읽어야 합니다.

## 7. 이 결과는 의미가 있는가?

p값만으로 결론을 내리지 않고 네 가지를 함께 봅니다.

1. **통계적 유의성**: p < 0.05 (다중 비교는 Holm 보정)
2. **효과 크기**: 실무 기준(+20%)을 넘는가
3. **신뢰구간**: 게임 단위 95% 신뢰구간이 0(변화 없음)을 포함하지 않는가
4. **강건성**: 분석 조건(전후 비교 기간)을 바꿔도 결론이 유지되는가

In [ ]:
# 강건성 점검: 전후 비교 기간을 3일·7일·14일로 바꿔서 같은 분석을 반복합니다.
rows = []
for w_ in (3, 7, 14):
    s_df, _ = build_samples(window=w_)
    e_, p_ = s_df[s_df['kind'] == 'event'], s_df[s_df['kind'] == 'placebo']
    if len(e_) < 10 or len(p_) < 10:
        continue
    lo_, hi_ = cluster_ci(e_, B=500)
    rows.append(dict(비교기간_일=w_, 이벤트=len(e_), 중앙값_변화_pct=to_pct(e_['log2'].median()), 위약_pct=to_pct(p_['log2'].median()),
                     신뢰구간=f'{to_pct(lo_):+.0f}% ~ {to_pct(hi_):+.0f}%',
                     p_위약대비=stats.mannwhitneyu(e_['log2'], p_['log2'], alternative='greater').pvalue))
sens = pd.DataFrame(rows)
display(sens)
R['sensitivity'] = sens.to_dict('records')

# 요일 효과 점검: (a) 요일 보정 검색량으로 같은 분석, (b) 위약을 할인 시작 요일 분포에 맞춰 가중, (c) 요일별 비교
s_adj, _ = build_samples(window=7, tr=trend_adj)
e_a, p_a = s_adj[s_adj['kind'] == 'event'], s_adj[s_adj['kind'] == 'placebo']
lo_a, hi_a = cluster_ci(e_a, B=500)
w_dow = ev['date'].dt.dayofweek.value_counts(normalize=True)
pl_dow = pl['date'].dt.dayofweek
present = [d for d in range(7) if (pl_dow == d).any()]
w_sum = sum(w_dow.get(d, 0) for d in present)
pl_weighted = sum(w_dow.get(d, 0) / w_sum * pl.loc[pl_dow == d, 'log2'].median() for d in present)
lo_o, hi_o = cluster_ci(ev, B=500)
sens_dow = pd.DataFrame([
    dict(방법='원본 7일', 할인_pct=to_pct(ev['log2'].median()), 위약_pct=to_pct(pl['log2'].median()), 신뢰구간=f'{to_pct(lo_o):+.0f}% ~ {to_pct(hi_o):+.0f}%'),
    dict(방법='요일 보정 검색량', 할인_pct=to_pct(e_a['log2'].median()), 위약_pct=to_pct(p_a['log2'].median()), 신뢰구간=f'{to_pct(lo_a):+.0f}% ~ {to_pct(hi_a):+.0f}%'),
    dict(방법='위약을 시작 요일 분포에 맞춤', 할인_pct=to_pct(ev['log2'].median()), 위약_pct=to_pct(pl_weighted), 신뢰구간=f'{to_pct(lo_o):+.0f}% ~ {to_pct(hi_o):+.0f}%'),
])
print('요일 효과 점검 (전후 7일 비교)')
display(sens_dow.round(1))
R['sensitivity_dow'] = sens_dow.to_dict('records')
e_d, p_d = ev.assign(dow=ev['date'].dt.dayofweek), pl.assign(dow=pl_dow)
by_day = [dict(요일=DOW[d], 이벤트n=int((e_d['dow'] == d).sum()), 위약n=int((p_d['dow'] == d).sum()),
               이벤트_pct=to_pct(e_d.loc[e_d['dow'] == d, 'log2'].median()), 위약_pct=to_pct(p_d.loc[p_d['dow'] == d, 'log2'].median()))
          for d in range(7) if (e_d['dow'] == d).sum() >= 8 and (p_d['dow'] == d).sum() >= 8]
print('같은 요일끼리 비교 (이벤트·위약 각각 8건 이상인 요일만)')
display(pd.DataFrame(by_day).round(1))
R['sensitivity_by_day'] = by_day

if len(sens):
    fig, ax = plt.subplots(figsize=(6.5, 3.6))
    ax.plot(sens['비교기간_일'], sens['중앙값_변화_pct'], marker='o', color=BLUE, label='할인 시작')
    ax.plot(sens['비교기간_일'], sens['위약_pct'], marker='o', color=GRAY, label='위약')
    ax.axhline((PRACTICAL - 1) * 100, color=ORANGE, ls='--', label='실무 기준 +20%'); ax.axhline(0, color='k', lw=.8)
    ax.set_xticks(sens['비교기간_일']); ax.set_xlabel('전후 비교 기간(일)'); ax.set_ylabel('검색량 변화 중앙값(%)'); ax.legend(); ax.set_title('비교 기간을 바꿔도 결론이 같은가')
    savefig(fig, 'a2_9_sensitivity.png'); plt.show()

### 차트 해설
전후 비교 기간을 3일·7일·14일로 바꿔서 같은 분석을 반복한 결과입니다.

### 이 차트로 알 수 있는 것
- 파란 선이 모든 기간에서 회색 선(위약)과 실무 기준(주황)보다 위에 있으면 결과가 **분석 설정에 좌우되지 않는 강건한 결과**입니다.
- 아래 표 두 개는 **요일 효과 점검**입니다. 요일 보정 검색량이나 같은 요일의 위약과 비교해도 할인 효과가 비슷하게 남으면 요일 때문에 생긴 결과가 아닙니다.
- 기간에 따라 크게 달라지면 7일 기준 결과를 그대로 믿기 어려우므로 발표에서 "조건에 따라 달라진다"고 밝혀야 합니다.

In [ ]:
# 가설별 최종 판정표
def mark(status):
    return {'match': '✅ 일치', 'partial': '⚠️ 부분 일치', 'mismatch': '❌ 불일치', 'skip': '검정 안 함'}[status]

print('⚠️ 부분 일치 = 통계적으로 유의하고 방향도 맞지만, 효과 크기가 사전에 정한 실무 기준(+20%)에 못 미침')
h1r, h2r, h3r = R['h1'], R['h2'], R['h3']
summary = pd.DataFrame([
    ['H1 할인이 시작되면 관심이 늘어난다', f"{h1r['median_pct']:+.0f}% (위약 {h1r['placebo_median_pct']:+.0f}%)", f"p={max(h1r['p_wilcoxon'], h1r['p_mwu']):.1e}", f"{h1r['ci_pct'][0]:+.0f}% ~ {h1r['ci_pct'][1]:+.0f}%", mark(h1r['status'])],
    ['H2 할인율이 클수록 관심이 더 늘어난다', f"Spearman rho={h2r['spearman_rho']:.2f}", f"p={h2r['spearman_p']:.1e}" + (f", KW p={h2r['kruskal_p']:.1e}" if h2r['kruskal_p'] is not None else ''), '구간별 표 참고',
     mark(h2r['status'])],
    ['H3 출시 화제성은 3주 안에 절반으로 식는다',
     (f"KM 중앙값 {h3r['km_median_days']:.0f}일" if h3r and h3r['km_median_days'] is not None else '중앙값 미도달') if h3r else '-',
     f"p={h3r['p_binom']:.3f}" if h3r else '-', f"n={h3r['n']}" if h3r else '-', mark(h3r['status'] if h3r else 'skip')],
], columns=['가설', '핵심 값', '검정', '95% 신뢰구간', '판정'])
summary

## 8. 결론 — 가설 대비 결과와 인사이트·제안

아래 문장은 위 분석 결과를 **그대로 불러와** 만들어집니다. 판정이 "불일치"이면 그 가설은 데이터가 지지하지 않는다고 발표에서 솔직하게 말하고, 제안은 일치한 부분만 근거로 삼습니다.

In [ ]:
h1r, h2r, h3r = R['h1'], R['h2'], R['h3']
pk, pr, dw = R['peak_adj'], R['peak_raw'], R['dow']
tbl = [r for r in h2r['table'] if r['중앙값_변화'] != -999]
L = []
L.append('■ 가설 대비 결과')
h1_note = f" 다만 효과 크기는 사전에 정한 실무 기준(+{(PRACTICAL - 1) * 100:.0f}%)에는 못 미침." if h1r['status'] == 'partial' else ''
L.append(f"  H1 할인 → 관심 증가: {mark(h1r['status'])}. 할인 시작 후 7일 검색량은 시작 전보다 중앙값 {h1r['median_pct']:+.0f}% "
         f"(할인 없는 날 {h1r['placebo_median_pct']:+.0f}%, 신뢰구간 {h1r['ci_pct'][0]:+.0f}~{h1r['ci_pct'][1]:+.0f}%, n={h1r['n']}).{h1_note}")
if h2r['status'] == 'match':
    L.append(f"  H2 할인율↑ → 관심↑: {mark('match')}. 위약보다 유의하고 +20%를 넘는 가장 낮은 구간은 [{h2r['min_bin']}] (순위상관 rho={h2r['spearman_rho']:.2f}).")
else:
    bins_txt = ', '.join(f"{r['구간']} {r['중앙값_변화']:+.0f}%(n={r['n']})" for r in tbl)
    tail = (f" 위약보다 유의하게 늘어난 구간은 {h2r['n_sig_bins']}/{h2r['n_tested_bins']}개 → 할인 자체는 관심을 올리지만 할인율이 클수록 더 오르지는 않음." if h2r['n_sig_bins'] > 0 else '')
    L.append(f"  H2 할인율↑ → 관심↑: {mark(h2r['status'])}(또는 표본 부족). 상관 rho={h2r['spearman_rho']:.2f}(p={h2r['spearman_p']:.3f}), 구간별 변화 중앙값 [{bins_txt}].{tail}")
if h3r:
    km_txt = f"{h3r['km_median_days']:.0f}일" if h3r['km_median_days'] is not None else '관측 기간 안에서 도달하지 않음'
    L.append(f"  H3 출시 화제성 3주 내 반감: {mark(h3r['status'])}. 반감 중앙값 {km_txt}, 21일 안에 절반이 된 게임 {h3r['share_within']:.0%} (n={h3r['n']}, 제외 {h3r['excluded']}).")
else:
    L.append('  H3 출시 화제성: 표본 부족으로 검정하지 않음.')

L.append('\n■ 퍼블리셔에게 제안 (할인 전략)')
flat = (h2r['status'] == 'mismatch' and h2r['kruskal_p'] is not None and h2r['kruskal_p'] >= ALPHA)
if h2r['status'] == 'match':
    L.append(f"  - 한국 유저의 관심을 움직이려면 최소 [{h2r['min_bin']}] 이상 깎아야 효과가 관측됨. 그보다 얕은 할인은 보통 날과 구분되는 관심 증가가 없었음.")
elif h1r['status'] in ('match', 'partial') and flat and h2r['n_sig_bins'] > 0 and tbl:
    low = tbl[0]
    L.append(f"  - 할인은 한국 검색 관심을 올림(+{h1r['median_pct']:.0f}%). 그러나 할인율 구간 사이의 차이는 확인되지 않음(Kruskal p={h2r['kruskal_p']:.2f}; 가장 얕은 구간 [{low['구간']}]도 {low['중앙값_변화']:+.0f}%, n={low['n']}{' — 표본이 작아 단정하기 어려움' if low['n'] < 30 else ''}).")
    L.append("    → '관심 확대'가 목적이라면 깊은 할인이 더 큰 관심을 만든다는 근거는 없음. 단, 검색 관심이 구매·매출로 이어지는지는 이 분석에서 측정하지 않았으므로 판매 데이터와 함께 판단해야 함.")
elif h1r['status'] in ('match', 'partial'):
    L.append('  - 할인 자체는 관심을 올리지만, 어느 할인율부터 효과가 나는지는 이 표본으로는 단정할 수 없음 → 표본(게임·기간)을 늘려 재분석 필요.')
else:
    L.append('  - 이 표본에서는 할인과 국내 검색 관심의 연결이 확인되지 않음 → 할인율 제안은 보류하고 다른 지표(판매량 등)와 함께 봐야 함.')

L.append('\n■ 광고 노출 시점의 근거')
if dw['chi2_p'] < ALPHA:
    L.append(f"  - 요일 효과 주의: 할인 시작의 {dw['top_share']:.0%}가 {dw['top_day']}요일에 몰려 있음(위약과 요일 분포 차이 p={dw['chi2_p']:.1e}). "
             f"원본 곡선의 정점({pr['peak_day']}일차)은 주말 같은 요일 효과가 섞인 값일 수 있어 요일 보정본으로 판단함.")
if pk['peak_clear']:
    L.append(f"  - 요일 보정 후에도 시작 후 {pk['peak_day']}일차(지수 {pk['peak_index']:.0f})가 첫 주 평균보다 {PEAK_MARGIN}포인트 이상 높은 정점으로 나타남. 첫 주 평균 {pk['week1_pct']:+.0f}%, 둘째 주 평균 {pk['week2_pct']:+.0f}% (시작 전 7일 수준 대비).")
else:
    L.append(f"  - 요일 보정 후에는 뚜렷한 정점이 없음: 할인 후 첫 주 평균 {pk['week1_pct']:+.0f}%, 둘째 주 평균 {pk['week2_pct']:+.0f}% (시작 전 7일 수준 대비). "
             "→ 특정 요일에 맞춘 광고보다 **할인 첫 주 전체**에 노출을 두는 것이 근거에 맞음.")
L.append('  - (인과가 아닌 "함께 나타나는 현상" 기준)')
if h3r and h3r['km_median_days'] is not None:
    L.append(f"  - 출시 화제성은 중앙값 {h3r['km_median_days']:.0f}일 만에 정점의 절반으로 식음 → 출시 광고는 그 안에 집중, 이후는 할인 시점 광고로 전환.")

L.append('\n■ 서비스(겜누리) 적용')
L.append('  - "지금 할인하면 국내 관심이 얼마나 오르나"를 게임 상세·알림 문구의 근거로 사용. 알림은 할인 시작 직후(첫 주)에 보내는 것이 근거에 맞음.')
print('\n'.join(L))

import json as _json
(RESULT_DIR / 'summary.json').write_text(_json.dumps(R, ensure_ascii=False, indent=2, default=float), encoding='utf-8')
print('\n요약 저장:', RESULT_DIR / 'summary.json')

### 한계 (발표에서 먼저 밝힐 것)
| 한계 | 영향 | 대응 |
| --- | --- | --- |
| **인과가 아니라 함께 나타나는 현상** | 할인과 같은 날 일어난 다른 사건(DLC·방송·시즌 세일)의 효과가 섞일 수 있음 | 위약 날짜 대조, 출시 직후 제외, 표본을 모아 평균으로 판단 |
| 인기 게임 `TOP_N`개 위주 (한국어 리뷰 많은 순) | 인기 없는 게임에는 일반화 불가 | 발표에서 "인기 게임 기준"이라고 명시 |
| 네이버 검색은 한국 PC·모바일 검색만 반영 | 해외 유저·다른 검색 경로(유튜브, 커뮤니티) 미반영 | "국내 검색 관심"으로 한정해 표현 |
| 일반 단어 이름의 게임은 다른 검색이 섞임 | 변화율이 왜곡 | 3-1 점검 후 `EXCLUDE_IDS`/`KEYWORD_OVERRIDE`로 보정 |
| 가격 이력이 약 23개월 | 출시 화제성(H3)은 이 기간 안에 출시한 게임만 가능 → 표본 작음 | 표본 수·제외 수를 항상 같이 표기 |
| 할인율은 PC 스토어(ITAD)만 | 콘솔 할인은 반영 안 됨 | 분석 범위를 PC로 한정 |
| 신규 NAVER API HUB에서 키워드 그룹·과거 데이터 한도가 같은지 | 한도가 다르면 요청 설계 변경 | 호출은 확인함(문서상 요청당 키워드 그룹 5개). 일일 한도 수치는 문서에 없음 (미확인) |
| **할인 시작이 특정 요일(금·화 등)에 몰림** | 요일 효과(주말 검색 증가)가 "할인 효과"로 읽힐 수 있음 | H1은 월~일이 한 번씩 들어가는 전후 7일로 비교, 요일 보정·같은 요일 위약 비교(7장)로 재확인 |
| 1차 결과를 본 뒤 요일 보정·판정 표기·표본 확대를 추가함 | 사후 보강은 분석 유연성을 키움 | 가설·실무 기준(+20%)은 바꾸지 않았고, 추가한 사실을 2장에 명시. 게임은 리뷰 수 순으로만 선택 |